# Distributed-training GPU benchmark

Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. This notebook installs a separate Python 3.12 / PyTorch 2.6 environment, benchmarks the project's real Trainer on your assigned GPU, and produces a downloadable result archive. It uses synthetic data and no provider API keys.

The public repository is pinned to source commit `3f96b82778688f27f87f710592b1912103939921`. Only benchmark instrumentation is overlaid; training source and configs remain unchanged. Runs on one GPU measure CUDA training, not multi-GPU scaling. bf16 is included only when supported. Keep the runtime connected until the final cell finishes; save the notebook with outputs included.

In [ ]:
import datetime, json, os, pathlib, signal, subprocess, sys, time
from IPython.display import Markdown, display
DT_ROOT = pathlib.Path('/content/distributed-training-benchmark')
DT_ROOT.mkdir(exist_ok=True)
DT_PROJECT = DT_ROOT / 'project'
DT_ENV = DT_ROOT / 'venv'
DT_RESULTS = DT_ROOT / ('results-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
DT_RESULTS.mkdir()
DT_COMMIT = '3f96b82778688f27f87f710592b1912103939921'
hardware = subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True)
(DT_RESULTS/'nvidia-smi.txt').write_text(hardware.stdout)
print(hardware.stdout)
print('Results:', DT_RESULTS)

## Install the supported runtime

Dependencies are installed into a dedicated environment, so an existing Colab PyTorch import does not need a kernel restart. This cell downloads several GB of CUDA libraries and may take a few minutes.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.13.1'], check=True)
DT_UV_ENV = dict(os.environ, UV_CACHE_DIR=str(DT_ROOT/'cache'), UV_PYTHON_INSTALL_DIR=str(DT_ROOT/'python'))
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--managed-python', str(DT_ENV)], env=DT_UV_ENV, check=True)
DT_PY = str(DT_ENV/'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', DT_PY, 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'], env=DT_UV_ENV, check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', DT_PY, 'numpy==2.5.4', 'pyyaml==6.0.3', 'psutil==7.2.2'], env=DT_UV_ENV, check=True)
if not DT_PROJECT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/sdivyanshu90/build-your-own-distributed-training.git', str(DT_PROJECT)], check=True)
subprocess.run(['git', '-C', str(DT_PROJECT), 'checkout', DT_COMMIT], check=True)

## Install benchmark instrumentation

These helper files add explicit CPU/CUDA metadata, CUDA synchronization and memory counters, precision controls, resume checks, and short profiler traces. They do not change `src/` or model configs.

In [ ]:
DT_PATCHES = {'benchmarks/_common.py': '"""Shared helpers for the CPU/gloo benchmark harness.\n\nEvery benchmark is launched with ``torchrun`` (or plain ``python`` for world\nsize 1) and appends one JSON object per run to a JSONL file under\n``benchmarks/results/``. Nothing here is GPU specific: the harness measures what\nthe *code* does on CPU + gloo and says so in every record (``hardware`` field).\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport platform\nimport resource\nimport sys\nfrom typing import Any\n\nsys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nimport torch  # noqa: E402\nimport torch.distributed as dist  # noqa: E402\n\nfrom src.config import TrainingConfig  # noqa: E402\n\nREPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nRESULTS_DIR = os.path.join(REPO_ROOT, "benchmarks", "results")\n\n# Named model sizes used by the benchmarks (all fp32, gloo-friendly).\nMODEL_PRESETS: dict[str, dict[str, int]] = {\n    "tiny": {"vocab_size": 256, "d_model": 64, "n_layers": 2, "n_heads": 4, "n_kv_heads": 2,\n                 "ffn_hidden_size": 128, "max_seq_len": 64},\n    "small": {"vocab_size": 4096, "d_model": 256, "n_layers": 4, "n_heads": 8, "n_kv_heads": 4,\n                  "ffn_hidden_size": 704, "max_seq_len": 128},\n    "mid": {"vocab_size": 8192, "d_model": 512, "n_layers": 8, "n_heads": 8, "n_kv_heads": 4,\n                "ffn_hidden_size": 1408, "max_seq_len": 128},\n}\n\n\ndef apply_overrides(cfg: TrainingConfig, *, preset: str, seq_len: int | None,\n                    micro_batch_size: int | None, grad_accum: int | None,\n                    tp: int, sharding: str | None = None) -> TrainingConfig:\n    """Turn the ``test_tiny`` config into a benchmark config (in place)."""\n    for k, v in MODEL_PRESETS[preset].items():\n        setattr(cfg.model, k, v)\n    if seq_len:\n        cfg.data.seq_len = seq_len\n    cfg.model.max_seq_len = max(cfg.model.max_seq_len, cfg.data.seq_len)\n    if micro_batch_size:\n        cfg.data.micro_batch_size = micro_batch_size\n    if grad_accum:\n        cfg.grad_accum_steps = grad_accum\n    cfg.parallel.tp_size = tp\n    if sharding:\n        cfg.parallel.sharding_strategy = sharding\n    cfg.data.global_batch_size = 0\n    cfg.eval_interval = cfg.save_interval = cfg.profile_steps = 0\n    cfg.log_interval = 10**9\n    cfg.backend = "gloo"\n    cfg.scheduler.max_steps = cfg.max_steps = 10**6  # constant-ish LR over the run\n    cfg.scheduler.warmup_steps = 5\n    return cfg\n\n\ndef peak_rss_mb() -> float:\n    """Peak resident set size of this process in MiB (Linux: ru_maxrss is KiB)."""\n    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0\n\n\ndef tensor_bytes(t: Any) -> int:\n    if t is None:\n        return 0\n    if hasattr(t, "to_local"):\n        t = t.to_local()\n    return int(t.numel() * t.element_size())\n\n\ndef state_bytes(model: torch.nn.Module, optimizer: torch.optim.Optimizer) -> dict[str, int]:\n    """Bytes of parameters / gradients / optimizer state *resident on this rank*."""\n    params = sum(tensor_bytes(p) for p in model.parameters())\n    grads = sum(tensor_bytes(p.grad) for p in model.parameters())\n    opt = 0\n    for st in optimizer.state.values():\n        for v in st.values():\n            if torch.is_tensor(v):\n                opt += tensor_bytes(v)\n    return {"params": params, "grads": grads, "optimizer": opt}\n\n\ndef hardware_info() -> dict[str, Any]:\n    cpu = "unknown"\n    try:\n        with open("/proc/cpuinfo") as f:\n            for line in f:\n                if line.startswith("model name"):\n                    cpu = line.split(":", 1)[1].strip()\n                    break\n    except OSError:\n        pass\n    backend = dist.get_backend() if dist.is_initialized() else "gloo"\n    cuda_run = backend == "nccl"\n    return {\n        "platform": platform.platform(),\n        "cpu": cpu,\n        "cpu_count": os.cpu_count(),\n        "torch": torch.__version__,\n        "python": platform.python_version(),\n        "backend": backend,\n        "device": "cuda" if cuda_run else "cpu",\n        "cuda": torch.version.cuda if cuda_run else None,\n        "gpu": torch.cuda.get_device_name() if cuda_run else None,\n        "loadavg_1m_at_end": round(os.getloadavg()[0], 2),\n        "omp_num_threads": os.environ.get("OMP_NUM_THREADS"),\n        "torch_num_threads": torch.get_num_threads(),\n    }\n\n\ndef emit(record: dict[str, Any], out: str | None) -> None:\n    """Rank-0 only: print and append one JSON record."""\n    if dist.is_initialized() and dist.get_rank() != 0:\n        return\n    record = {**record, "hardware": hardware_info()}\n    line = json.dumps(record)\n    print(line, flush=True)\n    if out:\n        os.makedirs(os.path.dirname(out) or ".", exist_ok=True)\n        with open(out, "a") as f:\n            f.write(line + "\\n")\n\n\ndef gather_obj(obj: Any) -> list[Any]:\n    if not dist.is_initialized():\n        return [obj]\n    out: list[Any] = [None] * dist.get_world_size()\n    dist.all_gather_object(out, obj)\n    return out\n', 'benchmarks/bench_train.py': '"""Training throughput / memory / convergence benchmark (CPU + gloo).\n\nLaunch (world size = nproc_per_node; parallel layout from --tp, dp inferred)::\n\n    OMP_NUM_THREADS=3 torchrun --standalone --nproc_per_node=2 \\\n        benchmarks/bench_train.py --preset small --tp 1 --steps 20 --out benchmarks/results/train.jsonl\n\nReports, per run (one JSONL record, rank 0): median/mean step time, global\ntokens/sec, per-rank peak RSS (MiB), per-rank resident model-state bytes\n(params / grads / optimizer state), and optionally the per-step loss curve.\nIt drives the real ``Trainer`` + ``train_step`` -- the same code path as ``train.py``.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport statistics\nimport time\n\nimport torch\nfrom _common import REPO_ROOT, apply_overrides, emit, gather_obj, peak_rss_mb, state_bytes\n\nfrom src.config import TrainingConfig\nfrom src.parallelism.process_groups import destroy_distributed\nfrom src.training.loop import train_step\nfrom src.training.trainer import Trainer\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument("--preset", default="small", choices=["tiny", "small", "mid"])\n    p.add_argument("--tp", type=int, default=1)\n    p.add_argument("--backend", choices=["gloo", "nccl"], default="gloo")\n    p.add_argument("--dtype", choices=["float32", "bfloat16"], default="float32")\n    p.add_argument("--activation-checkpointing", action="store_true")\n    p.add_argument("--config", default=None, help="Use a complete model config instead of a preset")\n    p.add_argument("--sharding", default=None, help="FULL_SHARD | NO_SHARD | SHARD_GRAD_OP")\n    p.add_argument("--seq-len", type=int, default=None)\n    p.add_argument("--mbs", type=int, default=None, help="micro batch size per rank")\n    p.add_argument("--accum", type=int, default=None)\n    p.add_argument("--steps", type=int, default=20)\n    p.add_argument("--warmup", type=int, default=3)\n    p.add_argument("--lr", type=float, default=None)\n    p.add_argument("--curve", action="store_true", help="record per-step loss")\n    p.add_argument("--label", default="")\n    p.add_argument("--out", default=None)\n    a = p.parse_args()\n\n    cfg = TrainingConfig.from_yaml(a.config or f"{REPO_ROOT}/config/test_tiny.yaml")\n    if a.config:\n        if a.seq_len:\n            cfg.data.seq_len = a.seq_len\n        if a.mbs:\n            cfg.data.micro_batch_size = a.mbs\n        if a.accum:\n            cfg.grad_accum_steps = a.accum\n        cfg.parallel.tp_size = a.tp\n        cfg.data.global_batch_size = 0\n        cfg.data.num_workers = 0\n        cfg.eval_interval = cfg.save_interval = cfg.profile_steps = 0\n    else:\n        apply_overrides(cfg, preset=a.preset, seq_len=a.seq_len, micro_batch_size=a.mbs,\n                        grad_accum=a.accum, tp=a.tp, sharding=a.sharding)\n    cfg.backend = a.backend\n    cfg.parallel.param_dtype = a.dtype\n    cfg.parallel.buffer_dtype = a.dtype\n    cfg.parallel.reduce_dtype = "float32"\n    cfg.parallel.activation_checkpointing = a.activation_checkpointing\n    if a.lr:\n        cfg.optimizer.lr = a.lr\n        cfg.scheduler.min_lr = a.lr / 10\n    cfg.run_id = f"bench_{a.label or a.preset}"\n    gpu_type = "cpu" if a.backend == "gloo" else "unlisted_gpu"\n    trainer = Trainer(cfg, gpu_type=gpu_type)\n    ctx = trainer.ctx\n    rss_after_build = peak_rss_mb()\n\n    times: list[float] = []\n    losses: list[float] = []\n    wall0 = 0.0\n    try:\n        for i in range(a.warmup + a.steps):\n            if i == a.warmup:\n                if ctx.device.type == "cuda":\n                    torch.cuda.synchronize(ctx.device)\n                    torch.cuda.reset_peak_memory_stats(ctx.device)\n                wall0 = time.perf_counter()\n            window, dw = trainer._next_window()\n            t0 = time.perf_counter()\n            m = train_step(trainer.model, window, trainer.optimizer, trainer.scheduler,\n                           ctx, cfg, gpu_type=gpu_type, data_wait_s=dw)\n            dt = time.perf_counter() - t0\n            if i >= a.warmup:\n                times.append(dt)\n            losses.append(m.loss)\n        wall = time.perf_counter() - wall0\n        tokens_step = (cfg.data.micro_batch_size * cfg.data.seq_len\n                       * cfg.grad_accum_steps * ctx.dims.dp_size)\n        sb = state_bytes(trainer.model, trainer.optimizer)\n        per_rank = gather_obj({"rank": ctx.rank, "peak_rss_mb": round(peak_rss_mb(), 1),\n                               "rss_after_build_mb": round(rss_after_build, 1),\n                               "cuda_peak_allocated_bytes": torch.cuda.max_memory_allocated(ctx.device) if ctx.device.type == "cuda" else 0,\n                               "cuda_peak_reserved_bytes": torch.cuda.max_memory_reserved(ctx.device) if ctx.device.type == "cuda" else 0,\n                               **sb})\n        rec = {\n            "bench": "train",\n            "label": a.label,\n            "preset": a.config or a.preset,\n            "backend": a.backend, "dtype": a.dtype,\n            "activation_checkpointing": a.activation_checkpointing,\n            "world_size": ctx.world_size, "tp": ctx.dims.tp_size, "dp": ctx.dims.dp_size,\n            "sharding": cfg.parallel.sharding_strategy,\n            "seq_len": cfg.data.seq_len, "micro_batch_size": cfg.data.micro_batch_size,\n            "grad_accum": cfg.grad_accum_steps, "tokens_per_step": tokens_step,\n            "n_params": cfg.model.num_parameters(),\n            "actual_n_params": sum(p.numel() for p in trainer.model.parameters()) if ctx.world_size == 1 else None,\n            "steps": a.steps, "warmup": a.warmup,\n            "step_time_median_s": round(statistics.median(times), 5),\n            "step_time_mean_s": round(statistics.fmean(times), 5),\n            "step_time_stdev_s": round(statistics.pstdev(times), 5),\n            "tokens_per_s": round(tokens_step * a.steps / wall, 1),\n            "tokens_per_s_median_step": round(tokens_step / statistics.median(times), 1),\n            "per_rank": per_rank,\n            "loss_first": losses[0], "loss_last": losses[-1],\n        }\n        if a.curve:\n            rec["loss_curve"] = [round(x, 5) for x in losses]\n        emit(rec, a.out)\n    finally:\n        destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n', 'benchmarks/bench_resume.py': '"""Checkpoint cost + resume-exactness benchmark (CPU + gloo).\n\nInside one ``torchrun`` job (world size = FSDP degree, tp=1) this:\n\n1. trains ``--steps`` steps continuously (run A) recording the per-step loss;\n2. trains ``--steps//2`` steps (run B), saves a sharded checkpoint (timed, sized);\n3. builds a fresh ``Trainer`` with ``resume_from`` (run C; timed load), trains the\n   remaining half and compares C\'s per-step losses with A\'s second half.\n\nReported: save seconds, load seconds, bytes on disk, max |loss_A - loss_C| over the\nsecond half, and the full curves so the claim can be inspected, not trusted.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport shutil\nimport tempfile\nimport time\n\nfrom _common import REPO_ROOT, apply_overrides, emit\n\nfrom src.checkpointing.checkpoint import save_checkpoint\nfrom src.config import TrainingConfig\nfrom src.parallelism.process_groups import destroy_distributed\nfrom src.training.loop import train_step\nfrom src.training.trainer import Trainer\n\n\ndef _dir_bytes(path: str) -> int:\n    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)\n\n\ndef _cfg(a: argparse.Namespace, run_id: str, ckpt_dir: str) -> TrainingConfig:\n    cfg = TrainingConfig.from_yaml(f"{REPO_ROOT}/config/test_tiny.yaml")\n    apply_overrides(cfg, preset=a.preset, seq_len=None, micro_batch_size=None,\n                    grad_accum=None, tp=1)\n    cfg.backend = a.backend\n    cfg.parallel.param_dtype = cfg.parallel.buffer_dtype = a.dtype\n    cfg.parallel.reduce_dtype = "float32"\n    cfg.max_steps = cfg.scheduler.max_steps = a.steps\n    cfg.scheduler.warmup_steps = 3\n    cfg.checkpoint_dir = ckpt_dir\n    cfg.run_id = run_id\n    return cfg\n\n\ndef _drive(t: Trainer, n: int) -> list[float]:\n    out = []\n    for _ in range(n):\n        w, dw = t._next_window()\n        out.append(train_step(t.model, w, t.optimizer, t.scheduler, t.ctx, t.config,\n                              gpu_type="cpu", data_wait_s=dw).loss)\n        t.step += 1\n    return out\n\n\ndef main() -> None:\n    p = argparse.ArgumentParser()\n    p.add_argument("--preset", default="small", choices=["tiny", "small", "mid"])\n    p.add_argument("--steps", type=int, default=30)\n    p.add_argument("--backend", choices=["gloo", "nccl"], default="gloo")\n    p.add_argument("--dtype", choices=["float32", "bfloat16"], default="float32")\n    p.add_argument("--out", default=None)\n    a = p.parse_args()\n    half = a.steps // 2\n    # Shared tmp dir across ranks: rank 0 creates it, path broadcast via env-free file.\n    import torch.distributed as dist\n    tmp_holder = [tempfile.mkdtemp(prefix="dt_bench_ckpt_") if int(os.environ.get("RANK", 0)) == 0 else None]\n    from src.parallelism.process_groups import init_distributed\n    init_distributed(a.backend)\n    dist.broadcast_object_list(tmp_holder, src=0)\n    tmp = tmp_holder[0]\n    try:\n        A = Trainer(_cfg(a, "A", tmp), gpu_type="cpu")\n        curve_a = _drive(A, a.steps)\n        B = Trainer(_cfg(a, "B", tmp), gpu_type="cpu")\n        _drive(B, half)\n        dist.barrier()\n        t0 = time.perf_counter()\n        ckpt = save_checkpoint(B.model, B.optimizer, B.scheduler, B.step, B.config, B.ctx)\n        save_s = time.perf_counter() - t0\n        size = _dir_bytes(ckpt)\n        cfg_c = _cfg(a, "B", tmp)\n        cfg_c.resume_from = ckpt\n        dist.barrier()\n        t0 = time.perf_counter()\n        C = Trainer(cfg_c, gpu_type="cpu")  # includes model build + load\n        build_and_load_s = time.perf_counter() - t0\n        curve_c = _drive(C, a.steps - half)\n        diffs = [abs(x - y) for x, y in zip(curve_a[half:], curve_c, strict=True)]\n        emit({\n            "bench": "resume", "preset": a.preset, "world_size": B.ctx.world_size,\n            "backend": a.backend, "dtype": a.dtype,\n            "dp": B.ctx.dims.dp_size, "steps": a.steps, "resume_at": half,\n            "n_params": B.config.model.num_parameters(),\n            "ckpt_save_s": round(save_s, 4), "trainer_build_plus_load_s": round(build_and_load_s, 4),\n            "ckpt_bytes_total": size, "max_abs_loss_diff": max(diffs),\n            "bit_identical_steps": sum(1 for d in diffs if d == 0.0), "compared_steps": len(diffs),\n            "loss_continuous": [round(x, 6) for x in curve_a],\n            "loss_resumed_second_half": [round(x, 6) for x in curve_c],\n        }, a.out)\n    finally:\n        dist.barrier()\n        if dist.get_rank() == 0:\n            shutil.rmtree(tmp, ignore_errors=True)\n        destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n', 'benchmarks/bench_profile.py': '"""Capture a short real training trace; profiling timings are separate from throughput."""\nfrom __future__ import annotations\n\nimport argparse\nimport gzip\nfrom pathlib import Path\n\nimport torch\nfrom _common import REPO_ROOT, apply_overrides, emit\n\nfrom src.config import TrainingConfig\nfrom src.observability.profiler import communication_fraction, text_summary\nfrom src.parallelism.process_groups import destroy_distributed\nfrom src.training.loop import train_step\nfrom src.training.trainer import Trainer\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--backend", choices=["gloo", "nccl"], default="gloo")\n    parser.add_argument("--out-dir", required=True)\n    parser.add_argument("--dtype", choices=["float32", "bfloat16"], default="bfloat16")\n    args = parser.parse_args()\n    out = Path(args.out_dir)\n    cfg = TrainingConfig.from_yaml(f"{REPO_ROOT}/config/test_tiny.yaml")\n    apply_overrides(cfg, preset="small", seq_len=128, micro_batch_size=4, grad_accum=2, tp=1)\n    cfg.backend = args.backend\n    if args.backend == "nccl":\n        cfg.parallel.param_dtype = cfg.parallel.buffer_dtype = args.dtype\n    trainer = Trainer(cfg, gpu_type="unlisted_gpu")\n    ctx = trainer.ctx\n    try:\n        for _ in range(5):\n            window, wait = trainer._next_window()\n            train_step(trainer.model, window, trainer.optimizer, trainer.scheduler, ctx, cfg, gpu_type="unlisted_gpu", data_wait_s=wait)\n        activities = [torch.profiler.ProfilerActivity.CPU]\n        if ctx.device.type == "cuda":\n            activities.append(torch.profiler.ProfilerActivity.CUDA)\n        with torch.profiler.profile(activities=activities, record_shapes=True, profile_memory=True) as prof:\n            for _ in range(3):\n                window, wait = trainer._next_window()\n                train_step(trainer.model, window, trainer.optimizer, trainer.scheduler, ctx, cfg, gpu_type="unlisted_gpu", data_wait_s=wait)\n        if ctx.is_rank0:\n            stem = f"profile-{args.backend}"\n            path = out / f"{stem}.json"\n            prof.export_chrome_trace(str(path))\n            with path.open("rb") as source, gzip.open(str(path) + ".gz", "wb") as dest:\n                dest.write(source.read())\n            path.unlink()\n            (out / f"{stem}.txt").write_text(text_summary(prof, 30))\n            breakdown = communication_fraction(prof)\n            ops = [{"name": e.key, "count": e.count, "self_cpu_us": e.self_cpu_time_total,\n                    "self_device_us": getattr(e, "self_device_time_total", 0)} for e in prof.key_averages()]\n            emit({"bench": "profile", "world_size": ctx.world_size, "backend": args.backend,\n                  "steps": 3, "communication_fraction": breakdown.comm_fraction,\n                  "total_attributed_us": breakdown.total_cuda_us,\n                  "communication_attributed_us": breakdown.comm_cuda_us, "operators": ops},\n                 str(out / "profiles.jsonl"))\n    finally:\n        destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n'}
for relative, source in DT_PATCHES.items():
    (DT_PROJECT/relative).write_text(source)
probe = subprocess.run([DT_PY, '-c', "import torch,json,platform; print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,'cuda':torch.version.cuda,'nccl':torch.cuda.nccl.version(),'gpu_count':torch.cuda.device_count(),'gpu':torch.cuda.get_device_name(0),'compute_capability':torch.cuda.get_device_capability(0),'gpu_memory_bytes':torch.cuda.get_device_properties(0).total_memory,'bf16_native':torch.cuda.is_bf16_supported(including_emulation=False),'bf16_including_emulation':torch.cuda.is_bf16_supported(including_emulation=True)}))"], check=True, capture_output=True, text=True)
DT_SYSTEM = json.loads(probe.stdout)
DT_SYSTEM['source_commit'] = subprocess.run(['git','-C',str(DT_PROJECT),'rev-parse','HEAD'], check=True, capture_output=True, text=True).stdout.strip()
assert DT_SYSTEM['source_commit'] == DT_COMMIT
import hashlib
(DT_RESULTS/'instrumentation.json').write_text(json.dumps({relative:hashlib.sha256(source.encode()).hexdigest() for relative,source in DT_PATCHES.items()}, indent=2))
freeze = subprocess.run([sys.executable,'-m','uv','pip','freeze','--python',DT_PY], check=True, capture_output=True, text=True, env=DT_UV_ENV)
(DT_RESULTS/'dependencies.txt').write_text(freeze.stdout)
assert DT_SYSTEM['gpu_count'] >= 1
(DT_RESULTS/'system.json').write_text(json.dumps(DT_SYSTEM, indent=2))
print(json.dumps(DT_SYSTEM, indent=2))
DT_DTYPES = ['float32'] + (['bfloat16'] if DT_SYSTEM['bf16_native'] else [])
print('Precision policies:', DT_DTYPES)
if not DT_SYSTEM['bf16_native']:
    print('BF16 acceleration is not native on this GPU; automatic benchmarks use float32.')

In [ ]:
DT_STATUS = []
def dt_run(name, script, arguments, seconds=600):
    command = [DT_PY, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=1', 'benchmarks/'+script, *arguments]
    start = time.monotonic()
    environment = dict(os.environ, CUDA_VISIBLE_DEVICES='0', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
    print('Running:', name, flush=True)
    with (DT_RESULTS/(name+'.log')).open('w') as log:
        process = subprocess.Popen(command, cwd=DT_PROJECT, env=environment, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            exit_code = process.wait(timeout=seconds)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
            exit_code = 124
    record = {'case':name, 'exit_code':exit_code, 'duration_s':round(time.monotonic()-start, 2), 'command':command}
    DT_STATUS.append(record)
    with (DT_RESULTS/'execution.jsonl').open('a') as output: output.write(json.dumps(record)+'\n')
    print(record)
    if exit_code: print((DT_RESULTS/(name+'.log')).read_text()[-5000:])

## Throughput: small and mid models

Each point has 10 warm-up and 50 timed steps, sequence 128, micro-batch 4, accumulation 2. Persistent model/Adam state stays fp32; bf16 is an autocast policy, selected automatically only when `is_bf16_supported(including_emulation=False)` is true. T4 uses float32; software/emulation support alone does not enable the bf16 cases. MFU is omitted for GPUs without a calibrated project peak.

In [ ]:
for dtype in DT_DTYPES:
    for preset in ['small','mid']:
        dt_run(f'cuda-{preset}-{dtype}', 'bench_train.py', ['--backend','nccl','--dtype',dtype,'--preset',preset,'--seq-len','128','--mbs','4','--steps','50','--warmup','10','--label',f'cuda_{preset}_{dtype}','--out',str(DT_RESULTS/'cuda.jsonl')])

## Activation-memory sweep and reduced 151.9M shape

The activation sweep uses micro-batch 1, accumulation 1, 5 warm-up + 20 timed steps. The larger model uses the architecture from `125m.yaml` but reduces sequence 1024→128, micro-batch 16→1, accumulation 4→1, and workers to 0. It has 3 warm-up + 10 timed steps; this is not the shipped full workload or a pretraining run.

In [ ]:
DT_DTYPE = 'bfloat16' if DT_SYSTEM['bf16_native'] else 'float32'
for sequence in [128,512,1024]:
    for checkpointing in [False,True]:
        arguments = ['--backend','nccl','--dtype',DT_DTYPE,'--preset','mid','--seq-len',str(sequence),'--mbs','1','--accum','1','--steps','20','--warmup','5','--label',f'cuda_mem_s{sequence}_ac{int(checkpointing)}','--out',str(DT_RESULTS/'cuda-memory.jsonl')]
        if checkpointing: arguments.append('--activation-checkpointing')
        dt_run(f'memory-s{sequence}-ac{int(checkpointing)}','bench_train.py',arguments)
for checkpointing in [False,True]:
    arguments = ['--backend','nccl','--dtype',DT_DTYPE,'--config','config/125m.yaml','--seq-len','128','--mbs','1','--accum','1','--steps','10','--warmup','3','--label',f'cuda_125m_s128_ac{int(checkpointing)}','--out',str(DT_RESULTS/'cuda-125m.jsonl')]
    if checkpointing: arguments.append('--activation-checkpointing')
    dt_run(f'125m-ac{int(checkpointing)}','bench_train.py',arguments)

## Resume exactness and profiler

The resume test compares 15 post-resume losses with a continuous 30-step run. A trace without GPU kernel timing events is unavailable for device-time attribution; raw zeros do not prove zero communication.

In [ ]:
for dtype in DT_DTYPES:
    dt_run('resume-'+dtype,'bench_resume.py',['--backend','nccl','--dtype',dtype,'--preset','small','--steps','30','--out',str(DT_RESULTS/'cuda-resume.jsonl')])
dt_run('profile','bench_profile.py',['--backend','nccl','--dtype',DT_DTYPE,'--out-dir',str(DT_RESULTS)])

## Save and download results

Save this notebook with outputs included, then share its Viewer link. The archive contains raw measurements, per-case logs, runtime metadata, and profiler artifacts. All cases are local to this Colab runtime; a failure remains in the execution record.

In [ ]:
import hashlib, shutil
rows = [json.loads(line) for line in (DT_RESULTS/'cuda.jsonl').read_text().splitlines()] if (DT_RESULTS/'cuda.jsonl').exists() else []
summary = '| Run | Tokens/s | Peak allocated MiB |\n| --- | --- | --- |\n'
for row in rows:
    summary += f"| {row['label']} | {row['tokens_per_s']} | {row['per_rank'][0]['cuda_peak_allocated_bytes']/1024**2:.1f} |\n"
display(Markdown(summary))
failed = [row['case'] for row in DT_STATUS if row['exit_code']]
print('Failed cases:', failed)
(DT_RESULTS/'sha256.txt').write_text(''.join(hashlib.sha256(path.read_bytes()).hexdigest()+'  '+path.name+'\n' for path in sorted(DT_RESULTS.iterdir()) if path.is_file() and path.name!='sha256.txt'))
DT_ARCHIVE = shutil.make_archive(str(DT_RESULTS), 'zip', DT_RESULTS)
print('Result archive:', DT_ARCHIVE)
from google.colab import files
files.download(DT_ARCHIVE)